# 07 — Score live flights + build alternative-flight recommender

Applies the fitted feature pipeline (from 04) to `api_silver_flights`, loads both
champion models from Unity Catalog, ensembles pre-departure and in-flight predictions,
and writes:

- `flight_delay_predictions` — one row per scored flight
- `alternative_flight_recommendations` — top 5 lower-risk alternatives per flight,
same route, ±3-hour window, ≥10% probability improvement

In [0]:
import sys
sys.path.append("..")

import mlflow
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array
from pyspark.ml.feature import VectorSlicer
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from src import config

mlflow.set_registry_uri(config.MLFLOW_REGISTRY_URI)

## Load fitted feature pipeline + API silver

In [0]:
feature_pipeline = PipelineModel.load(f"{config.ARTIFACT_VOLUME}/feature_pipeline")
api_silver = spark.table(config.API_SILVER)
print(f"Rows to score: {api_silver.count():,}")

Rows to score: 54


## Apply feature transformation

In [0]:
api_with_hours = api_silver.withColumn(
    "dep_hour", (F.col("crs_dep_time") / 100).cast("int")
).withColumn(
    "arr_hour", (F.col("crs_arr_time") / 100).cast("int")
)

transformed = feature_pipeline.transform(api_with_hours.na.fill(0).na.fill("UNKNOWN"))
n_features = len(transformed.select(vector_to_array("features")).first()[0])
pre_indices = [i for i in range(n_features) if i != 11]

slicer_pre = VectorSlicer(inputCol="features", outputCol="features_pre", indices=pre_indices)
scored_input = slicer_pre.transform(transformed)

## Load champions from UC

In [0]:
rf_pre = mlflow.spark.load_model(f"models:/{config.MODEL_RF_PRE}@{config.CHAMPION_ALIAS}", dfs_tmpdir=config.ARTIFACT_VOLUME)
gbt_pre = mlflow.spark.load_model(f"models:/{config.MODEL_GBT_PRE}@{config.CHAMPION_ALIAS}", dfs_tmpdir=config.ARTIFACT_VOLUME)
rf_in = mlflow.spark.load_model(f"models:/{config.MODEL_RF_IN}@{config.CHAMPION_ALIAS}", dfs_tmpdir=config.ARTIFACT_VOLUME)
gbt_in = mlflow.spark.load_model(f"models:/{config.MODEL_GBT_IN}@{config.CHAMPION_ALIAS}", dfs_tmpdir=config.ARTIFACT_VOLUME)

## Score both variants and ensemble

In [0]:
def _prob1(df, model, features_col, out_col):
    # Extract the classifier from the PipelineModel (single stage)
    classifier = model.stages[0]
    # Transform with the classifier using the specified features column
    df2 = classifier.copy().setFeaturesCol(features_col).transform(df)
    return df2.withColumn(out_col, vector_to_array("probability")[1]).drop(
        "rawPrediction", "probability", "prediction"
    )

scored = scored_input
scored = _prob1(scored, rf_pre, "features_pre", "prob_rf_pre")
scored = _prob1(scored, gbt_pre, "features_pre", "prob_gbt_pre")
scored = _prob1(scored, rf_in, "features", "prob_rf_in")
scored = _prob1(scored, gbt_in, "features", "prob_gbt_in")

scored = scored.withColumn("prob_pre", (F.col("prob_rf_pre") + F.col("prob_gbt_pre")) / 2.0)
scored = scored.withColumn("prob_in", (F.col("prob_rf_in") + F.col("prob_gbt_in")) / 2.0)

def _risk(col):
    return (
        F.when(col >= 0.7, F.lit("High"))
         .when(col >= 0.5, F.lit("Medium"))
         .otherwise(F.lit("Low"))
    )

scored = scored.withColumn("risk_pre", _risk(F.col("prob_pre")))
scored = scored.withColumn("risk_in", _risk(F.col("prob_in")))

## Predictions table

In [0]:
predictions = scored.select(
    F.current_timestamp().alias("prediction_timestamp"),
    "airline_name", "airline_code", "fl_number",
    F.concat_ws(" → ", "origin_airport_code", "destination_airport_code").alias("route"),
    "origin_airport_code", "destination_airport_code",
    "flight_date", "crs_dep_time", "crs_arr_time", "dep_delay",
    (F.col("prob_pre") * 100).alias("prob_delay_pre_pct"),
    F.col("risk_pre").alias("risk_pre_departure"),
    (F.col("prob_in") * 100).alias("prob_delay_in_pct"),
    F.col("risk_in").alias("risk_in_flight"),
    (F.col("prob_rf_pre") * 100).alias("rf_pre_pct"),
    (F.col("prob_gbt_pre") * 100).alias("gbt_pre_pct"),
    (F.col("prob_rf_in") * 100).alias("rf_in_pct"),
    (F.col("prob_gbt_in") * 100).alias("gbt_in_pct"),
)

(
    predictions.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config.PREDICTIONS)
)
print(f"Wrote {predictions.count()} predictions → {config.PREDICTIONS}")

Wrote 54 predictions → workspace.flights.flight_delay_predictions


## Alternative-flight recommender
Same origin/dest, ±3 hours from the original flight, at least 10 percentage points
lower delay probability. Ranks by improvement + risk bonus. Top 5 per flight.

In [0]:
candidates = predictions.selectExpr(
    "airline_name AS alt_airline",
    "airline_code AS alt_airline_code",
    "fl_number AS alt_flight",
    "origin_airport_code",
    "destination_airport_code",
    "flight_date",
    "crs_dep_time AS alt_crs_dep_time",
    "prob_delay_pre_pct AS alt_prob_delay_pct",
    "risk_pre_departure AS alt_risk",
    "dep_delay AS alt_dep_delay",
)

joined = (
    predictions.alias("orig")
    .join(
        candidates.alias("alt"),
        (F.col("orig.origin_airport_code") == F.col("alt.origin_airport_code"))
        & (F.col("orig.destination_airport_code") == F.col("alt.destination_airport_code"))
        & (F.col("orig.flight_date") == F.col("alt.flight_date"))
        & (F.col("orig.fl_number") != F.col("alt.alt_flight"))
        & (F.abs(F.col("orig.crs_dep_time") - F.col("alt.alt_crs_dep_time")) <= 300),  # ±3h in HHMM ≈ 300
    )
    .withColumn(
        "improvement_pct",
        F.col("orig.prob_delay_pre_pct") - F.col("alt.alt_prob_delay_pct"),
    )
    .filter(F.col("improvement_pct") >= 10.0)
    .withColumn(
        "risk_bonus",
        F.when(F.col("alt.alt_risk") == "Low", 20)
         .when(F.col("alt.alt_risk") == "Medium", 10)
         .otherwise(0),
    )
    .withColumn("recommendation_score", F.col("improvement_pct") + F.col("risk_bonus"))
)

window = Window.partitionBy(
    "orig.airline_code", "orig.fl_number", "orig.flight_date"
).orderBy(F.col("recommendation_score").desc())

recommendations = (
    joined.withColumn("recommendation_rank", F.row_number().over(window))
    .filter(F.col("recommendation_rank") <= 5)
    .select(
        F.col("orig.airline_name").alias("original_airline"),
        F.col("orig.fl_number").alias("original_flight"),
        F.col("orig.origin_airport_code").alias("origin"),
        F.col("orig.destination_airport_code").alias("destination"),
        F.col("orig.flight_date").alias("flight_date"),
        F.col("orig.prob_delay_pre_pct").alias("original_delay_prob"),
        F.col("orig.dep_delay").alias("original_dep_delay"),
        F.col("alt.alt_airline").alias("alternative_airline"),
        F.col("alt.alt_airline_code").alias("alternative_airline_code"),
        F.col("alt.alt_flight").alias("alternative_flight"),
        F.col("alt.alt_prob_delay_pct").alias("alternative_delay_prob"),
        F.col("alt.alt_risk").alias("alternative_risk_level"),
        F.col("alt.alt_dep_delay").alias("alternative_dep_delay"),
        "improvement_pct",
        "recommendation_score",
        "recommendation_rank",
        F.current_timestamp().alias("recommendation_timestamp"),
    )
)

(
    recommendations.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config.ALTERNATIVES)
)
print(f"Wrote {recommendations.count()} recommendations → {config.ALTERNATIVES}")

Wrote 0 recommendations → workspace.flights.alternative_flight_recommendations


In [0]:
%sql
SELECT * FROM workspace.flights.flight_delay_predictions LIMIT 20;

prediction_timestamp,airline_name,airline_code,fl_number,route,origin_airport_code,destination_airport_code,flight_date,crs_dep_time,crs_arr_time,dep_delay,prob_delay_pre_pct,risk_pre_departure,prob_delay_in_pct,risk_in_flight,rf_pre_pct,gbt_pre_pct,rf_in_pct,gbt_in_pct
2026-09-01T02:09:39.158Z,Virgin Atlantic,VS,5139,ATL → LAX,ATL,LAX,2026-08-31,1947,20,6.0,22.88739442157751,Low,14.129960095256832,Low,20.207834909778413,25.566953933376613,17.638295532698745,10.621624657814921
2026-09-01T02:09:39.158Z,LATAM Airlines,LA,7952,ATL → LAX,ATL,LAX,2026-08-31,1947,20,6.0,22.88739442157751,Low,14.129960095256832,Low,20.207834909778413,25.566953933376613,17.638295532698745,10.621624657814921
2026-09-01T02:09:39.158Z,Aeromexico,AM,4629,ATL → LAX,ATL,LAX,2026-08-31,1947,20,6.0,22.88739442157751,Low,14.129960095256832,Low,20.207834909778413,25.566953933376613,17.638295532698745,10.621624657814921
2026-09-01T02:09:39.158Z,Air France,AF,2973,ATL → LAX,ATL,LAX,2026-08-31,1947,20,6.0,23.80355876576796,Low,14.01158017174297,Low,20.46171274036003,27.145404791175885,17.401535685671025,10.621624657814921
2026-09-01T02:09:39.158Z,WestJet,WS,5503,ATL → LAX,ATL,LAX,2026-08-31,2330,355,0.0,22.42644897472077,Low,12.768434710445254,Low,20.207834909778413,24.64506303966313,17.638295532698745,7.898573888191763
2026-09-01T02:09:39.158Z,Virgin Atlantic,VS,4383,ATL → LAX,ATL,LAX,2026-08-31,2330,355,0.0,22.42644897472077,Low,12.768434710445254,Low,20.207834909778413,24.64506303966313,17.638295532698745,7.898573888191763
2026-09-01T02:09:39.158Z,LATAM Airlines,LA,6627,ATL → LAX,ATL,LAX,2026-08-31,2330,355,0.0,22.42644897472077,Low,12.768434710445254,Low,20.207834909778413,24.64506303966313,17.638295532698745,7.898573888191763
2026-09-01T02:09:39.158Z,Korean Air,KE,7085,ATL → LAX,ATL,LAX,2026-08-31,2330,355,0.0,22.42644897472077,Low,12.768434710445254,Low,20.207834909778413,24.64506303966313,17.638295532698745,7.898573888191763
2026-09-01T02:09:39.158Z,Aeromexico,AM,4623,ATL → LAX,ATL,LAX,2026-08-31,2330,355,0.0,22.42644897472077,Low,12.768434710445254,Low,20.207834909778413,24.64506303966313,17.638295532698745,7.898573888191763
2026-09-01T02:09:39.158Z,Air France,AF,6843,ATL → LAX,ATL,LAX,2026-08-31,2330,355,0.0,22.42644897472077,Low,12.768434710445254,Low,20.207834909778413,24.64506303966313,17.638295532698745,7.898573888191763
